<a href="https://colab.research.google.com/github/akashskypatel/NeurCross/blob/main/NeurCross_Training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install git+https://github.com/akashskypatel/NeurCross.git
!pip install trimesh


In [ ]:
import kagglehub
import os

# List of Kaggle dataset identifiers
KAGGLE_DATASET_IDS = [
    "balraj98/modelnet40-princeton-3d-object-dataset",
    "mrbiid/meshy-ai-800-glb-3d-assets-categorised-and-labelled",
    "programmer3/cadcae-design-dataset",
    "timfuger/part-processing-dataset",
    "chienru/mask2-3d",
    "sc0v1n0/3d-models",
    "lukaszfuszara/thingi10k-name-and-category",
    "amytai/nutritionverse-3d",
    "mrbiid/meshy-ai-363-ply-creatures-labelled",
]

# This list will store the actual local paths where the datasets are downloaded
LOCAL_DOWNLOADED_PATHS = []

for dataset_id in KAGGLE_DATASET_IDS:
  # kagglehub.dataset_download returns the local path to the downloaded dataset
  local_path = kagglehub.dataset_download(dataset_id)
  LOCAL_DOWNLOADED_PATHS.append(local_path)

# You can now use LOCAL_DOWNLOADED_PATHS to access the files
print("Local paths of downloaded datasets:")
for p in LOCAL_DOWNLOADED_PATHS:
    print(p)

In [ ]:
from huggingface_hub import snapshot_download
import os

# Use the huggingface_hub API to download the dataset repository
print("Downloading dataset using Hugging Face API...")
repo_id = "akashskypatel/NeurCross"
extract_dir = snapshot_download(
    repo_id=repo_id,
    repo_type="dataset",
    allow_patterns=["directional-data.zip"]
)

# Since snapshot_download returns the folder, we still need to unzip the specific file if it's compressed
zip_file_path = os.path.join(extract_dir, "directional-data.zip")
final_extract_path = "/content/directional_data_hf"

if os.path.exists(zip_file_path):
    import zipfile
    print(f"Extracting {zip_file_path} to {final_extract_path}...")
    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(final_extract_path)

    # Add the extracted folder to the global path list
    if final_extract_path not in LOCAL_DOWNLOADED_PATHS:
        LOCAL_DOWNLOADED_PATHS.append(final_extract_path)
        print(f"Added {final_extract_path} to LOCAL_DOWNLOADED_PATHS")
else:
    # If the files are already loose in the repo, just add the extract_dir
    if extract_dir not in LOCAL_DOWNLOADED_PATHS:
        LOCAL_DOWNLOADED_PATHS.append(extract_dir)
        print(f"Added {extract_dir} to LOCAL_DOWNLOADED_PATHS")

In [ ]:
allowed_extensions = {'.obj', '.off', '.stl', '.ply', '.glb'}
mesh_files = []

# Iterate through the actual local paths where datasets were downloaded
for base_path in LOCAL_DOWNLOADED_PATHS:
    print(f"\nSearching in: {base_path}")
    found_files_in_current_path = False
    for root, dirs, files in os.walk(base_path):
        for file in files:
            file_extension = os.path.splitext(file)[1].lower()
            if file_extension in allowed_extensions:
                mesh_files.append(os.path.join(root, file))
                print(f"file: {os.path.join(root, file)} added")
                found_files_in_current_path = True
    if not found_files_in_current_path:
        print(f"  No matching files found in {base_path}.")

print(f"\nFound {len(mesh_files)} mesh files in total.")

In [ ]:
import neurcross
import pathlib
import subprocess
import os
import sys

out_dir = "/home/train/neurocross"
pathlib.Path(out_dir).mkdir(parents=True, exist_ok=True)

# Re-check mesh_files
allowed_extensions = {'.obj', '.off', '.stl', '.ply', '.glb'}
mesh_files = []
if 'LOCAL_DOWNLOADED_PATHS' in globals():
    for base_path in LOCAL_DOWNLOADED_PATHS:
        for root, dirs, files in os.walk(base_path):
            for file in files:
                if os.path.splitext(file)[1].lower() in allowed_extensions:
                    mesh_files.append(os.path.join(root, file))

target_mesh = None
for f_path in mesh_files:
    if f_path.endswith('.obj') or f_path.endswith('.off'):
        target_mesh = f_path
        break

if not target_mesh and mesh_files:
    target_mesh = mesh_files[0]

if target_mesh:
    print(f"Using mesh file: {target_mesh}")
    # Parameters for stability
    n_samples = 10000

    command = [
        "python",
        "-m",
        "neurcross",
        "train-quad-mesh",
        "--data_path", target_mesh,
        "--out_dir", out_dir,
        "--n_samples", str(n_samples)
    ]

    print(f"Executing command: {' '.join(command)}")

    # Use Popen to stream output in real-time
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1
    )

    print("--- Real-time Training Output ---")
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()  # Ensure output is printed immediately

    process.wait()
    if process.returncode != 0:
        print(f"\nProcess failed with return code {process.returncode}")
else:
    print("No compatible mesh files found.")